In [1]:
%pip install dbt-core dbt-duckdb duckdb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

# Définition des dossiers
GOLD_DIR = "."
MODELS_DIR = os.path.join(GOLD_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# 1. Création de la configuration dbt (dbt_project.yml)
dbt_project = """
name: 'bank_reviews_gold'
version: '1.0.0'
config-version: 2
profile: 'bank_reviews_profile'
model-paths: ["models"]
clean-targets: ["target", "dbt_packages"]
"""
with open(os.path.join(GOLD_DIR, "dbt_project.yml"), "w") as f:
    f.write(dbt_project)

# 2. Création du profil de connexion pour DuckDB (profiles.yml)
profiles = """
bank_reviews_profile:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: 'bank_reviews_gold.duckdb'
      threads: 4
"""
with open(os.path.join(GOLD_DIR, "profiles.yml"), "w") as f:
    f.write(profiles)

# 3. Création du Modèle SQL pour les KPI
sql_model = """
{{ config(materialized='table') }}

with silver_data as (
    select * from read_parquet('../silver/*.parquet')
)

select
    agency_name,
    count(*) as total_reviews,
    round(avg(rating), 2) as avg_rating,
    sum(case when sentiment = 'Positive' then 1 else 0 end) as positive_reviews,
    sum(case when sentiment = 'Negative' then 1 else 0 end) as negative_reviews,
    sum(case when sentiment = 'Neutral' then 1 else 0 end) as neutral_reviews
from silver_data
group by agency_name
"""
with open(os.path.join(MODELS_DIR, "agency_kpis.sql"), "w") as f:
    f.write(sql_model)

print("✅ Fichiers de configuration dbt créés avec succès !")
print("🚀 Lancement de dbt pour générer la table Gold...")

# 4. Exécution de dbt via la ligne de commande Python
!python -m dbt run --profiles-dir . --project-dir .

✅ Fichiers de configuration dbt créés avec succès !
🚀 Lancement de dbt pour générer la table Gold...


/mnt/c/Users/USER/Desktop/bank_reviews_lakehouse/airflow_env/bin/python: No module named dbt.__main__; 'dbt' is a package and cannot be directly executed


In [3]:
import duckdb

# 1. Connexion (ou création) de la base de données Gold
con = duckdb.connect('bank_reviews_gold.duckdb')

# 2. Création de la table avec une requête SQL directe
print("⚙️ Création de la table Gold en cours...")

con.execute("""
CREATE OR REPLACE TABLE agency_kpis AS 
SELECT
    agency_name,
    count(*) as total_reviews,
    round(avg(rating), 2) as avg_rating,
    sum(case when sentiment = 'Positive' then 1 else 0 end) as positive_reviews,
    sum(case when sentiment = 'Negative' then 1 else 0 end) as negative_reviews,
    sum(case when sentiment = 'Neutral' then 1 else 0 end) as neutral_reviews
FROM read_parquet('../silver/*.parquet')
GROUP BY agency_name
""")

# 3. Créer aussi une table avec toutes les reviews détaillées
con.execute("""
CREATE OR REPLACE TABLE reviews_detail AS
SELECT * FROM read_parquet('../silver/*.parquet')
""")

print("✅ Tables Gold créées avec succès !")

# 4. Affichage du résultat
print("📊 Tableau de bord Gold (KPIs par agence) :")
display(con.execute("SELECT * FROM agency_kpis").df())

# 5. Fermeture propre de la connexion
con.close()

⚙️ Création de la table Gold en cours...
✅ Tables Gold créées avec succès !
📊 Tableau de bord Gold (KPIs par agence) :


,agency_name,total_reviews,avg_rating,positive_reviews,negative_reviews,neutral_reviews
0,Banque Populaire,13,1.54,2.0,3.0,8.0
1,AL AKHDAR BANK Siège,54,3.69,11.0,7.0,36.0
2,Attijariwafa Bank,52,1.90,4.0,14.0,34.0
3,Crédit Agricole,6,4.17,3.0,0.0,3.0
4,Banque Populaire Ibn Khaldoun,14,1.36,1.0,7.0,6.0


In [4]:
import duckdb
import os
from datetime import datetime

# =============================================================
# EXPORT AUTOMATIQUE POUR POWER BI
# Cette cellule exporte les données en CSV à chaque exécution.
# Compatible avec Airflow : les fichiers sont toujours au même
# emplacement, donc Power BI les actualise automatiquement.
# =============================================================

# Dossier de sortie pour Power BI
POWERBI_DIR = os.path.join('.', 'powerbi_export')
os.makedirs(POWERBI_DIR, exist_ok=True)

con = duckdb.connect('bank_reviews_gold.duckdb')

# --- 1. Export des KPIs par agence ---
df_kpis = con.execute("SELECT * FROM agency_kpis").df()
kpis_path = os.path.join(POWERBI_DIR, 'agency_kpis.csv')
df_kpis.to_csv(kpis_path, index=False, encoding='utf-8-sig')
print(f"✅ KPIs exportés : {kpis_path} ({len(df_kpis)} agences)")

# --- 2. Export des avis détaillés ---
df_reviews = con.execute("SELECT * FROM reviews_detail").df()
reviews_path = os.path.join(POWERBI_DIR, 'reviews_detail.csv')
df_reviews.to_csv(reviews_path, index=False, encoding='utf-8-sig')
print(f"✅ Détails exportés : {reviews_path} ({len(df_reviews)} avis)")

# --- 3. Fichier metadata (date de dernière mise à jour) ---
meta_path = os.path.join(POWERBI_DIR, 'last_update.txt')
with open(meta_path, 'w') as f:
    f.write(f"Dernière mise à jour : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Nombre d'agences : {len(df_kpis)}\n")
    f.write(f"Nombre total d'avis : {len(df_reviews)}\n")

con.close()

print(f"\n📁 Fichiers Power BI prêts dans : {os.path.abspath(POWERBI_DIR)}")
print("\n💡 Dans Power BI :")
print("   1. Obtenir les données → Texte/CSV")
print(f"   2. Sélectionnez : {os.path.abspath(kpis_path)}")
print(f"   3. Et aussi : {os.path.abspath(reviews_path)}")
print("   4. Pour actualiser : Accueil → Actualiser 🔄")

✅ KPIs exportés : ./powerbi_export/agency_kpis.csv (5 agences)
✅ Détails exportés : ./powerbi_export/reviews_detail.csv (139 avis)

📁 Fichiers Power BI prêts dans : /mnt/c/Users/USER/Desktop/bank_reviews_lakehouse/datalake/gold/powerbi_export

💡 Dans Power BI :
   1. Obtenir les données → Texte/CSV
   2. Sélectionnez : /mnt/c/Users/USER/Desktop/bank_reviews_lakehouse/datalake/gold/powerbi_export/agency_kpis.csv
   3. Et aussi : /mnt/c/Users/USER/Desktop/bank_reviews_lakehouse/datalake/gold/powerbi_export/reviews_detail.csv
   4. Pour actualiser : Accueil → Actualiser 🔄
